<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Aguja en el Pajar y el Guardia Dormilón 🪡
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/08%20-%20Temas%20Avanzados%20de%20ML/Para%20Dummies/03_Desbalanceados_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 03 del Módulo 08 (Datasets desbalanceados)](../03_Datasets_Desbalanceados.ipynb). Hay problemas donde lo que buscas es **rarísimo**: un ladrón entre mil visitantes, una enfermedad grave entre miles de pacientes. Un modelo que aprende de esos datos **tiende a ignorar lo raro**. Aquí vemos por qué, cómo se arregla, y una **trampa** muy común al arreglarlo. Solo usamos *NumPy*, *scikit-learn* y gráficas.

Al terminar podrás explicar:
1. Por qué «**98 % de aciertos**» puede significar «**no sirve para nada**» (el guardia dormilón).
2. Tres remedios: **copiar** casos raros, **inventar casos parecidos** (mezclando dos conocidos) y **bajar el listón de sospecha**.
3. La **trampa del examen con preguntas copiadas**: si copias casos **antes** de separar práctica y examen, el examen **se llena de copias** y te pones una nota falsa.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../03_Datasets_Desbalanceados.ipynb). Antes de esto, [cómo explicar las decisiones de un modelo](02_Explicabilidad_Dummies.ipynb).

---
## 1. El guardia dormilón 😴

Un museo recibe **1000 visitantes** y **20 son ladrones** (el 2 %). El guardia, cansado, **nunca levanta la alarma** y dice siempre «este visitante es honesto». ¿Cuánto acierta? **El 98 %**: casi todos los visitantes **sí** son honestos. ¿Cuántos ladrones atrapa? **Ninguno**. Esto se llama la **paradoja de la exactitud**: cuando casi todo es de un tipo, **acertar mucho** es **fácil** y **no demuestra nada**. Por eso, en estos problemas hay que mirar **cuántos de los raros se atrapan** (el *recall*) y **cuántas alarmas son falsas**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(1)
visitantes = 1000
es_ladron = np.zeros(visitantes, dtype=int)
es_ladron[rng.choice(visitantes, 20, replace=False)] = 1                         # 20 ladrones de 1000

guardia_dormilon = np.zeros(visitantes, dtype=int)                               # siempre dice "honesto"
acierto = np.mean(guardia_dormilon == es_ladron)
atrapados = np.sum((guardia_dormilon == 1) & (es_ladron == 1))

print(f"El guardia dormilón acierta el {acierto:.0%} de las veces… y atrapa {atrapados} de {es_ladron.sum()} ladrones.")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.bar(["Aciertos totales", "Ladrones atrapados"], [acierto, atrapados / es_ladron.sum()], color=["#16a34a", "#dc2626"])
ax.set_ylim(0, 1.1); ax.set_ylabel("Fracción"); ax.set_title("98 % de aciertos… y ningún ladrón atrapado", fontweight="bold", fontsize=10)
for i, v in enumerate([acierto, atrapados / es_ladron.sum()]):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center")
plt.tight_layout(); plt.show()
assert acierto == 0.98 and atrapados == 0

---
---
**Lección:** el número que más se publica (los aciertos) **no dice nada** cuando una clase es rara. Pregunta siempre: **¿cuántos de los casos raros se detectan?** y **¿cuántas falsas alarmas hay?**

## 2. Tres remedios para el guardia 🛠️

Ahora entrenamos a un **guardia de verdad** (un modelo sencillo) con **2000 visitantes** de los que solo **5 %** son ladrones, y lo probamos con **20 000 visitantes nuevos**. Cada visitante se describe por **dos números** (por ejemplo, «nerviosismo» y «tiempo que se queda»). Comparamos:

| Remedio | Cómo se explica | Qué hace |
|---|---|---|
| **Nada** | El guardia aprende tal cual | Casi nunca da la alarma |
| **Copiar ladrones** (*sobremuestreo*) | Fotocopiar las fichas de los ladrones conocidos hasta que haya tantos como honestos | Más peso a los raros (¡pero son **copias exactas**!) |
| **Inventar ladrones parecidos** (*SMOTE*) | Elegir **dos ladrones conocidos que se parecen** y crear un visitante **a medio camino** entre ellos | Más variedad que copiar |
| **Bajar el listón** (*mover el umbral*) | No cambiar los datos: dar la alarma desde que haya **5 % de sospecha** (la proporción real de ladrones) en vez de 50 % | Cambia **dónde se corta**, no lo que se aprendió |

In [ ]:
rng = np.random.default_rng(2)

def visitantes_nuevos(n, prevalencia=0.05):
    y = (rng.random(n) < prevalencia).astype(int)
    X = rng.normal(0, 1, (n, 2)) + np.where(y[:, None] == 1, [1.4, 1.0], [0.0, 0.0])      # los ladrones se parecen un poco a los honestos
    return X, y

X_ent, y_ent = visitantes_nuevos(2000)
X_test, y_test = visitantes_nuevos(20000)

def copiar_raros(X, y):
    """Fotocopia (con repeticion) a los raros hasta igualar a los comunes."""
    raros = np.where(y == 1)[0]
    extra = rng.choice(raros, (y == 0).sum() - len(raros), replace=True)
    return np.vstack([X, X[extra]]), np.r_[y, y[extra]]

def inventar_parecidos(X, y, k=5):
    """SMOTE a mano: punto nuevo = un raro + un trozo del camino hacia uno de sus k raros mas cercanos."""
    R = X[y == 1]
    d = ((R[:, None, :] - R[None, :, :]) ** 2).sum(axis=2)
    vecinos = np.argsort(d, axis=1)[:, 1:k + 1]                                  # los k mas cercanos (sin contarse a si mismo)
    cuantos = (y == 0).sum() - len(R)
    i = rng.integers(0, len(R), cuantos)
    j = vecinos[i, rng.integers(0, k, cuantos)]
    nuevos = R[i] + rng.random((cuantos, 1)) * (R[j] - R[i])                      # a medio camino (una fraccion al azar del camino)
    return np.vstack([X, nuevos]), np.r_[y, np.ones(cuantos, dtype=int)]

def evaluar(nombre, modelo, umbral=0.5):
    p = modelo.predict_proba(X_test)[:, 1]
    alarma = p > umbral
    atrapados = (alarma & (y_test == 1)).sum() / (y_test == 1).sum()
    falsas = (alarma & (y_test == 0)).sum() / max(alarma.sum(), 1)
    return {"remedio": nombre, "ladrones atrapados": atrapados, "de cada 100 alarmas, falsas": 100 * falsas, "aciertos totales": np.mean(alarma == y_test)}

filas = []
base = LogisticRegression().fit(X_ent, y_ent)
filas.append(evaluar("Nada", base))
filas.append(evaluar("Copiar ladrones", LogisticRegression().fit(*copiar_raros(X_ent, y_ent))))
Xs, ys = inventar_parecidos(X_ent, y_ent)
filas.append(evaluar("Inventar parecidos (SMOTE)", LogisticRegression().fit(Xs, ys)))
filas.append(evaluar("Bajar el listón (alarma desde 5 %)", base, umbral=0.05))      # umbral = proporcion real de ladrones
print(f"{'Remedio':<32}{'Ladrones atrapados':>20}{'Falsas por cada 100 alarmas':>30}{'Aciertos totales':>20}")
for f in filas:
    print(f"{f['remedio']:<32}{f['ladrones atrapados']:>20.0%}{f['de cada 100 alarmas, falsas']:>30.0f}{f['aciertos totales']:>20.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
axes[0].scatter(*X_ent[y_ent == 0].T, s=8, color="#94a3b8", alpha=0.5, label="honestos")
axes[0].scatter(*X_ent[y_ent == 1].T, s=26, color="#dc2626", label="ladrones conocidos")
axes[0].scatter(*Xs[len(y_ent):].T, s=6, color="#16a34a", marker="x", alpha=0.6, label="ladrones «inventados» (SMOTE)")
axes[0].set_xlabel("nerviosismo"); axes[0].set_ylabel("tiempo que se queda"); axes[0].legend(fontsize=8); axes[0].set_title("SMOTE: nuevos casos entre ladrones parecidos", fontweight="bold", fontsize=10)
axes[1].bar([f["remedio"].split(" (")[0].replace(" ", "\n", 1) for f in filas], [f["ladrones atrapados"] for f in filas], color=["#94a3b8", "#f59e0b", "#16a34a", "#2563eb"])
axes[1].set_ylabel("Fracción de ladrones atrapados"); axes[1].set_title("Todos los remedios atrapan más ladrones…", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert filas[0]["ladrones atrapados"] < 0.3 and min(f["ladrones atrapados"] for f in filas[1:]) > 0.5
assert abs(filas[3]["ladrones atrapados"] - filas[1]["ladrones atrapados"]) < 0.08           # bajar el liston ~ copiar
assert all(f["aciertos totales"] < filas[0]["aciertos totales"] for f in filas[1:])           # ¡a costa de bajar los aciertos totales y de mas falsas alarmas!

---
---
**Lección:** con los tres remedios el guardia **atrapa muchos más ladrones**, pero **a costa de más falsas alarmas** (y de bajar los «aciertos totales», ¡por eso esa cifra engaña!). Ojo con dos cosas: **(1)** *Bajar el listón* hasta la proporción real de ladrones logra **prácticamente lo mismo** que copiar o inventar (≈ 78 % atrapados y ≈ 80 falsas por cada 100 alarmas) **sin tocar los datos**, así que conviene probarlo **primero**; **(2)** *copiar* y *SMOTE* no son magia: el modelo **no aprende nada nuevo**, solo cambia **cuánto le pesan** los raros. Cuánta falsa alarma aceptas **depende de cuánto cuesta cada error** (¿un ladrón que se escapa o un visitante molestado?).

## 3. La trampa del examen con preguntas copiadas 📝

Un estudiante (un modelo muy memorioso: «mira el caso más parecido y copia su respuesta») debe prepararse con casos de **ladrones** y se examina con otros casos. Si **copias los casos raros 20 veces antes de separar** «práctica» y «examen», las **copias exactas caen en las dos partes**: el estudiante ve en el examen **la misma pregunta que ya memorizó**. La nota sale **espectacular**… y **falsa**. Lo correcto: **primero separar práctica y examen, y copiar solo la práctica**.

In [ ]:
rng = np.random.default_rng(3)
X, y = visitantes_nuevos(3000)

def un_vecino(X_ent, y_ent, X_nuevo):
    """El estudiante memorioso: copia la respuesta del caso de practica MAS PARECIDO."""
    d = ((X_nuevo[:, None, :] - X_ent[None, :, :]) ** 2).sum(axis=2)
    return y_ent[d.argmin(axis=1)]

def ladrones_atrapados(y_verdad, y_pred):
    return ((y_pred == 1) & (y_verdad == 1)).sum() / max((y_verdad == 1).sum(), 1)

resultados = {"mal": [], "bien": [], "verdad": []}
X_real, y_real = visitantes_nuevos(4000)                                          # el mundo real (nunca visto)
for rep in range(5):
    # MAL: copiar a los raros 20 veces ANTES de separar practica y examen
    raros = np.where(y == 1)[0]
    idx = np.r_[np.arange(len(y)), np.repeat(raros, 19)]
    Xc, yc = X[idx], y[idx]
    orden = rng.permutation(len(yc)); mitad = len(yc) // 2
    tr, te = orden[:mitad], orden[mitad:]
    resultados["mal"].append(ladrones_atrapados(yc[te], un_vecino(Xc[tr], yc[tr], Xc[te])))
    # BIEN: primero separar; copiar SOLO la practica; el examen queda intacto
    orden = rng.permutation(len(y)); mitad = len(y) // 2
    tr, te = orden[:mitad], orden[mitad:]
    raros_tr = tr[y[tr] == 1]
    idx_tr = np.r_[tr, np.repeat(raros_tr, 19)]
    resultados["bien"].append(ladrones_atrapados(y[te], un_vecino(X[idx_tr], y[idx_tr], X[te])))
    resultados["verdad"].append(ladrones_atrapados(y_real, un_vecino(X[idx_tr], y[idx_tr], X_real)))

medias = {k: np.mean(v) for k, v in resultados.items()}
fig, ax = plt.subplots(figsize=(7.8, 4.3))
ax.bar(["Copiar ANTES de separar\n(nota falsa)", "Separar y luego copiar\n(nota honesta)", "En el mundo real\n(la verdad)"], [medias["mal"], medias["bien"], medias["verdad"]], color=["#dc2626", "#16a34a", "#2563eb"])
ax.set_ylim(0, 1.1); ax.set_ylabel("Ladrones atrapados en el examen")
for i, v in enumerate([medias["mal"], medias["bien"], medias["verdad"]]):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center")
ax.set_title("Copiar antes de separar infla la nota", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
print(f"Nota falsa (copiar antes de separar): {medias['mal']:.0%} | nota honesta (separar y luego copiar): {medias['bien']:.0%} | mundo real: {medias['verdad']:.0%}")
assert medias["mal"] > 0.9 and medias["mal"] > medias["bien"] + 0.3 and abs(medias["bien"] - medias["verdad"]) < 0.15

---
---
**Lección:** copiar antes de separar da una nota casi perfecta (**~100 %**) que **no existe** en el mundo real (**~30 %**). La regla de oro: **separa primero lo que será el examen y haz los arreglos (copiar, inventar, rellenar…) solo con la parte de práctica.** El examen debe quedar **intacto**, con la proporción real de ladrones.

## Resumen en una frase 🎯

> **Cuando lo que buscas es raro, los aciertos totales engañan: hay que contar cuántos raros se atrapan y cuántas falsas alarmas hay; se puede ayudar al modelo copiando, inventando parecidos o bajando el listón (¡probando el listón primero!), y los arreglos se hacen solo con la parte de práctica, nunca antes de separar el examen.**

### Lo que aprendiste hoy:
- ✅ **El guardia dormilón** acierta 98 % y no atrapa a nadie: **la exactitud engaña** con clases raras.
- ✅ **Copiar** raros, **inventar parecidos (SMOTE)** y **bajar el listón** atrapan más raros **a costa de más falsas alarmas**; **bajar el listón** logra casi lo mismo **sin tocar los datos**.
- ✅ Copiar o inventar **no enseña nada nuevo**: solo cambia **cuánto pesan** los raros.
- ✅ **La trampa**: copiar **antes** de separar práctica y examen da una nota falsa (~100 % frente a ~30 % real). **Separa primero.**

> 🎓 **Siguiente paso:** el [cuaderno estándar](../03_Datasets_Desbalanceados.ipynb) implementa SMOTE **paso a paso con verificaciones**, mide la fuga con números y compara las estrategias con las métricas correctas; y luego, [el álbum con páginas que faltan (datos faltantes)](04_Datos_Faltantes_Dummies.ipynb).

---
##### 🛠️ Práctica 1: ¿Cuánto bajar el listón? Probar cuatro alarmas

Bajar el listón no es todo o nada: puedes dar la alarma desde **50 %**, **20 %**, **5 %** o **2 %** de sospecha. Usa el guardia entrenado `base` y la función `evaluar(nombre, modelo, umbral)`:

1. Haz la lista `umbrales = [0.5, 0.2, 0.05, 0.02]`.
2. Para cada umbral llama a `evaluar(...)` con `base` y ese umbral, y guarda los resultados en la lista `filas_u`.
3. Imprime, para cada umbral, cuántos ladrones se atrapan, cuántas de cada 100 alarmas son falsas y los aciertos totales.
4. Comprueba con `assert` que, al bajar el listón de 50 % a 2 %, se atrapan **más de 50 puntos porcentuales más** de ladrones, pero también **suben las falsas alarmas** y **bajan los aciertos totales**.

In [ ]:
# Escribe tu código aquí

# umbrales = [0.5, 0.2, 0.05, 0.02]
# filas_u = [evaluar(f"alarma desde {100 * u:g} %", base, umbral=...) for u in umbrales]
# for f in filas_u:
#     print(f["remedio"], f["ladrones atrapados"], f["de cada 100 alarmas, falsas"], f["aciertos totales"])


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
umbrales = [0.5, 0.2, 0.05, 0.02]
filas_u = [evaluar(f"alarma desde {100 * u:g} %", base, umbral=u) for u in umbrales]

print(f"{'Listón':<22}{'Ladrones atrapados':>20}{'Falsas por cada 100 alarmas':>30}{'Aciertos totales':>20}")
for f in filas_u:
    print(f"{f['remedio']:<22}{f['ladrones atrapados']:>20.0%}{f['de cada 100 alarmas, falsas']:>30.0f}{f['aciertos totales']:>20.1%}")

primero, ultimo = filas_u[0], filas_u[-1]
assert ultimo["ladrones atrapados"] > primero["ladrones atrapados"] + 0.5
assert ultimo["de cada 100 alarmas, falsas"] > primero["de cada 100 alarmas, falsas"]
assert ultimo["aciertos totales"] < primero["aciertos totales"]
print("Cuanto más bajo el listón, más ladrones atrapados... y más falsas alarmas: hay que decidir cuántas falsas alarmas se pueden aceptar.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Un hospital dice: «nuestro detector de una enfermedad rara acierta el 99.5 %». ¿Qué dos preguntas harías antes de creerle?
2. ¿Qué diferencia hay entre **copiar** los casos raros y **inventar casos parecidos**? ¿En qué se parecen (¿el modelo aprende algo realmente nuevo?)
3. Explica con tus palabras por qué **copiar antes de separar práctica y examen** da una nota falsa. ¿Cómo lo harías bien?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>